In [1]:
import pandas as pd
import os
from pathlib import Path
import unicodedata

In [2]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [3]:
def remove_accents(texto: str) -> str:
    nfd = unicodedata.normalize("NFD", texto)
    return "".join(c for c in nfd if unicodedata.category(c) != "Mn")

In [4]:
df_classified = pd.read_json(root / "classifier/resultados.jsonl", lines = True)

In [5]:
df_classified.head(2)

,id,tipo_ejercicio,confianza,error
0,fisica_campo_gravitatorio_2016_septiembre_ejer...,satélites,alta,NaN
1,fisica_campo_gravitatorio_2026_junio_ejercicio_a,verdadero / falso,alta,NaN


In [6]:
df_classified.loc[:, ["asignatura", "tema"]] = ["Física", "Campo gravitatorio"]

In [7]:
df_classified.confianza.value_counts()

confianza
alta     116
media      5
baja       2
Name: count, dtype: int64

Insertar nombre del tema en nombre archivo csv

In [8]:
df_classified.to_csv(root / "classifier/campo_gravitatorio_classified_exercises.csv",
          index = False, encoding="utf-8")

## Combinar enunciados con clasificaciones

Cargar enunciados

In [9]:
df_enun = pd.read_csv(root / "csv/ejercicios_clean.csv", encoding="utf-8")

In [10]:
df_enun.head(2)

,id,asignatura,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo,imagen
0,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio A,campo eléctrico y magnético,FISICA. 2025. JULIO. EJERCICIO a,Discuta la veracidad de las siguientes afirmac...,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...
1,fisica_campo_electrico_y_magnetico_2025_julio_...,Física,2025,Julio,Ejercicio B1,campo eléctrico y magnético,FISICA. 2025. JULIO. EJERCICIO b1,Dos conductores rectilíneos muy largos se disp...,source_pdf_files/Física/campo_eléctrico_magn...,enunciados_ejercicios/Física/campo_eléctrico...


### Inspeccionar clasificados

In [11]:
df_classified.columns

Index(['id', 'tipo_ejercicio', 'confianza', 'error', 'asignatura', 'tema'], dtype='str')

In [12]:
merged_df = pd.merge(df_enun,
         df_classified.loc[:, ["id", "tipo_ejercicio"]],
         on = "id", how = "left")


In [13]:
rearranged_cols = ['asignatura', 'año', 'convocatoria', 'ejercicio', 'tema',
                   'tipo_ejercicio','enunciado',
                   "id",
                   'info_ejercicio', 'archivo', 'imagen'
                   ]
merged_df = merged_df[rearranged_cols]

In [14]:
merged_df[merged_df.tema == df_classified.loc[0, "tema"].lower()].sample(5)

,asignatura,año,convocatoria,ejercicio,tema,tipo_ejercicio,enunciado,id,info_ejercicio,archivo,imagen
186,Física,2020,Junio,Ejercicio 1,campo gravitatorio,distribución_masas,a) i) ¿Puede ser nulo el campo gravitatorio en...,fisica_campo_gravitatorio_2020_junio_ejercicio_1,FISICA. 2020. JUNIO. EJERCICIO 1,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...
180,Física,2021,Reserva 2,Ejercicio A1,campo gravitatorio,plano_inclinado,a) Una partícula se mueve en un campo gravitat...,fisica_campo_gravitatorio_2021_reserva_2_ejerc...,FISICA. 2021. RESERVA 2. EJERCICIO A1,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...
163,Física,2022,Julio,Ejercicio A2,campo gravitatorio,distribución_masas,a) Dos cuerpos de masas m y 2m están separados...,fisica_campo_gravitatorio_2022_julio_ejercicio_a2,FISICA. 2022. JULIO. EJERCICIO A2,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...
139,Física,2024,Julio,Ejercicio A2,campo gravitatorio,plano_inclinado,a) Razone si son verdaderos los siguientes enu...,fisica_campo_gravitatorio_2024_julio_ejercicio_a2,FISICA. 2024. JULIO. EJERCICIO A2,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...
227,Física,2017,Reserva 2,"Ejercicio 1, opción B",campo gravitatorio,satélites,a) Supongamos que la Tierra reduce su radio a ...,fisica_campo_gravitatorio_2017_reserva_2_ejerc...,FISICA. 2017. RESERVA 2. EJERCICIO 1. OPCIÓN B,source_pdf_files/Física/campo_gravitatorio/20...,enunciados_ejercicios/Física/campo_gravitator...


Exportar enunciados clasificados

In [15]:
merged_df.to_csv(root / "csv/ejercicios_clasificados.csv",
                 index = False, encoding="utf-8")